---
# DNN : `spam` (avec `Keras` et `KerasTuner`)
---

## Packages

In [ ]:
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

from keras import callbacks, layers, models

from keras_tuner.tuners import RandomSearch

---
## 1. Données
---

### 1.1. Importation

In [ ]:
# En local :
directory = '/Users/vincentlefieux/Dropbox/Docs_ACADEMIQUE/Data/'

# Sur Google collab ou Onyxia (sur un répertoire temporaire) :
# directory = ''

# Sur Google collab (sur le drive) :
# from google.colab import drive
# drive.mount('/content/drive')
# directory = '/content/drive/MyDrive/Data/'

In [ ]:
data = pd.read_csv(directory + 'spam.csv',
                   header    = 0,
                   index_col = None,
                   sep       = ',',
                   decimal   = '.')

In [ ]:
data.info()

In [ ]:
data.head()

### 1.2. Gestion des données manquantes

In [ ]:
missing_percentage = data.isna().mean() * 100

print('MISSING VALUES :')
if missing_percentage[missing_percentage != 0].empty:
    print('No')
else:
    print(missing_percentage[missing_percentage != 0].sort_values(ascending=False))

### 1.3. Gestion des variables

In [ ]:
target = 'spam'

y = data[target]
X = data.drop(target, axis=1)

### 1.4. Création des échantillons de validation et test

In [ ]:
test_portion  = 1/5
valid_portion = 1/5

X_train_valid, X_test, y_train_valid, y_test = train_test_split(X, y, test_size=test_portion, random_state=1234)

X_train, X_valid, y_train, y_valid = train_test_split(X_train_valid, y_train_valid, test_size=valid_portion)

print('Dimensions de X_train :', X_train.shape)
print('Dimensions de X_valid :', X_valid.shape)
print('Dimensions de X_test  :', X_test.shape)

print('Dimensions de y_train :', y_train.shape)
print('Dimensions de y_valid :', y_valid.shape)
print('Dimensions de y_test  :', y_test.shape)

### 1.5. Normalisation des covariables

On normalise (centrage-réduction) les covariables :

In [ ]:
scaler = StandardScaler(with_mean=True, with_std=True)
scaler.fit(X_train)

X_train_norm = scaler.transform(X_train)
X_valid_norm = scaler.transform(X_valid)
X_test_norm  = scaler.transform(X_test)

---
## 2. DNN
---

### 2.1. Architecture et optimiseur

In [ ]:
dim_outputs = 1

n_layers_min = 1
n_layers_max = 3

n_units_min  = 32
n_units_max  = 128
n_units_step = 32

dropout_min  = 0.2
dropout_max  = 0.2
dropout_step = 0.1

activation_hidden = 'relu'
activation_output = 'sigmoid'

optimizer            = 'adam'
# learning_rate_values = [1e-3, 5e-4, 1e-4, 5e-5, 1e-5]
loss                 = 'binary_crossentropy'
metrics              = ['accuracy']

n_epochs_max = 200
batch_size   = 300
patience     = 10
verbose      = 1

def build_model(hp):
    model = models.Sequential()
    for i in range(hp.Int('num_layers', n_layers_min, n_layers_max)):
        model.add(layers.Dense(units = hp.Int('units_' + str(i+1),
                                              min_value = n_units_min,
                                              max_value = n_units_max,
                                              step      = n_units_step),
                               activation = activation_hidden,
                               name = 'Hidden_layer_' + str(i+1)))
        model.add(layers.Dropout(rate=hp.Float('dropout_' + str(i+1),
                                                min_value = dropout_min,
                                                max_value = dropout_max,
                                                step      = dropout_step)))
    model.add(layers.Dense(dim_outputs, activation=activation_output, name='Output_layer'))

    model.compile(optimizer = optimizer,
                  # optimizer = keras.optimizers.Adam(learning_rate=learning_rate),
                  loss      = loss,
                  metrics   = metrics)

    return model

### 2.2. Entraînement

In [ ]:
objective              = 'val_loss'
n_trials               = 50
n_executions_per_trial = 1
directory              = 'Project'
project_name           = 'Spam_Tuner'

monitor  = 'val_loss'
mode     = 'min'
patience = 10

batch_size   = 300
n_epochs_max = 200
verbose      = 0

tuner = RandomSearch(build_model,
                     objective            = objective,
                     max_trials           = n_trials,
                     executions_per_trial = n_executions_per_trial,
                     directory            = directory,
                     project_name         = project_name)

callback = callbacks.EarlyStopping(monitor              = monitor,
                                   mode                 = mode,
                                   patience             = patience,
                                   restore_best_weights = True)

tuner.search(X_train_norm,
             y_train,
             batch_size      = batch_size,
             epochs          = n_epochs_max,
             validation_data = (X_valid_norm, y_valid),
             callbacks       = [callback],
             verbose         = verbose)

In [ ]:
best_hp = tuner.get_best_hyperparameters()[0]

print(f"Nombre de couches : {best_hp.get('num_layers')}")

for i in range(1, best_hp.get('num_layers') + 1):
    print(f"Nombre de neurones couche {i} : {best_hp.get(f'units_{i}')}")

In [ ]:
best_model = tuner.get_best_models(num_models=1)[0]

### 2.3. Prévisions

In [ ]:
y_test_pred = best_model.predict(X_test_norm)
y_test_pred[0:5]

In [ ]:
y_test_pred_classes = (y_test_pred > 0.5).astype(int)
y_test_pred_classes[0:5]

In [ ]:
score_test = best_model.evaluate(X_test_norm, y_test, verbose=0)
print(f'Entropie test   : {score_test[0]:4.4f}')
print(f'Exactitude test : {score_test[1]:4.4f}')